# FORGE M5 — GPU smoke run (Gate E validation)

**Purpose:** measure real training throughput (tokens/sec) on free Kaggle GPU(s) so the M5 compute estimates (EXP-003, Gate E) can be trusted. Those estimates are currently *derived* (6ND ÷ T4 peak × 25–35% MFU) — this notebook replaces the guesswork with a measurement.

**Expected GPU budget:** < 30 minutes of GPU time. Kaggle sessions cap at 12h; weekly quota ~30 GPU-hours.

**What this does NOT prove:** loss on the toy byte-stream is a correctness fixture, not coding capability. GPU runs are also *not* expected to be bit-identical across runs (CUDA kernel nondeterminism) — the M3 bit-identical resume result was CPU-only and stays the reference. Here we validate **throughput and loss sanity only**.

**Code provenance:** the notebook checks out the exact pinned commit below — never a moving branch. Record the actual HEAD in the results JSON.

**Inputs you provide:** this notebook + your FORGE repo (GitHub push or uploaded zip — cell 3 covers both).

In [ ]:
import torch, subprocess, json, os, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch):", torch.version.cuda if torch.cuda.is_available() else None)
print("cudnn:", torch.backends.cudnn.version() if torch.cuda.is_available() else None)
n = torch.cuda.device_count()
print("visible GPUs:", n)
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"  gpu{i}: {props.name} | total VRAM {total/1e9:.1f} GB | free {free/1e9:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
      capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "NO CUDA — enable the GPU accelerator in notebook settings first"
print("ENV CHECK: CUDA is usable")


## Get the code

**Option A (recommended):** the repo is already on GitHub and `REPO_URL` is pre-filled in the cell below — just run it.

**Option B:** zip your forge repo, upload it as a Kaggle dataset (or use "Add data"), and set `ZIP_PATH` — the cell unzips it. (Commit pinning is only verifiable for Option A.)

In [ ]:
import os, shutil, subprocess, zipfile

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('command failed: ' + ' '.join(cmd) + '\n' + r.stderr[-2000:])
    return r.stdout.strip()

# --- Set these once, then Run All. Values are strings, so they need quotes:
# a bare URL (https://...) on its own line is a SyntaxError. ---
REPO_URL = "https://github.com/mohnish-ydv/forge"  # Option A: clone from GitHub
FORGE_COMMIT = "fa631226c43f2ce24df5c6793ec8491ec5baf4f3"  # pinned GitHub commit — never a moving branch
ZIP_PATH = ""  # Option B only: path to an uploaded forge zip, else ""

REPO_DIR = '/kaggle/working/forge'
if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        sh(['git', 'clone', REPO_URL, REPO_DIR])
        print('cloned', REPO_URL)
    else:
        print('repo dir already exists — reusing it')
elif ZIP_PATH:
    zipfile.ZipFile(ZIP_PATH).extractall(REPO_DIR)
    # flatten one level of nesting (a zip of a folder) if needed
    if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
        subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(os.path.join(REPO_DIR, d))]
        if len(subs) == 1 and os.path.isdir(os.path.join(REPO_DIR, subs[0], 'src')):
            for item in os.listdir(os.path.join(REPO_DIR, subs[0])):
                shutil.move(os.path.join(REPO_DIR, subs[0], item), os.path.join(REPO_DIR, item))
            os.rmdir(os.path.join(REPO_DIR, subs[0]))
    print('unzipped', ZIP_PATH)
else:
    raise SystemExit('Set REPO_URL (Option A) or ZIP_PATH (Option B) above, then re-run')
print('code dir:', REPO_DIR)


In [ ]:
os.chdir('/kaggle/working/forge')
actual = None
if REPO_URL:
    # (re-)enforce the pin even if a stale checkout already exists
    sh(['git', 'fetch', 'origin', FORGE_COMMIT])
    sh(['git', 'checkout', '--force', FORGE_COMMIT])
    sh(['git', 'clean', '-fdx', '-e', 'checkpoints'])  # drop stale untracked files, keep ckpts
    actual = sh(['git', 'rev-parse', 'HEAD'])
    assert actual.startswith(FORGE_COMMIT), f'pinned {FORGE_COMMIT} but HEAD is {actual}'
    print('pinned commit verified:', actual)
else:
    print('ZIP path: commit pinning not verifiable — record your repo state manually')
print('ACTUAL_HEAD=' + str(actual))

## Toy data (deterministic, sha-pinned)

Same generator as M3 (`src/toy_data.py`). ~2.5M tokens so the representative run doesn't just wrap a tiny file hundreds of times. This is a throughput fixture, not a corpus.

In [ ]:
import sys; sys.path.insert(0, 'src')
from toy_data import write_toy_dataset, load_ids
info = write_toy_dataset('data/toy/train.bin', n_sequences=65536)
ids = load_ids('data/toy/train.bin')
print('sha256:', info['sha256'])
print('tokens:', len(ids))
# n_sequences=65536 sha (M3's 1ea08995... is the 256-sequence default — different size, different file)
assert info['sha256'] == 'cab143199e42e84fadb51c75b5475dea370258a00bc3030da516c204bd78a8e7', 'toy data changed!'


## Run A — sanity: the exact M3 tiny model on CUDA

40 steps, 1.02M params. On CPU this gave loss 91.3 → 17.8 at ~4k tok/s. On GPU we expect the same loss trajectory (roughly — GPU nondeterminism means small numeric differences, not bit-identity) and much higher tok/s.

**No silent CPU fallback:** `--device cuda` (explicit) makes train.py raise if CUDA is unavailable, and the cell asserts the log line `device=cuda`.

In [ ]:
import subprocess
r = subprocess.run(['python', 'src/train.py', '--steps', '40', '--t-max', '40',
                    '--device', 'cuda', '--ckpt-dir', 'checkpoints/gpu_smoke_A',
                    '--log-every', '10', '--ckpt-every', '40'],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert 'device=cuda' in r.stdout, 'SILENT CPU FALLBACK? device=cuda not in log'
assert r.returncode == 0, 'Run A failed:\n' + r.stderr[-2000:]
print('RUN A OK — confirmed on CUDA (cuda:0)')


## Run B — representative: 25.3M params (30M class), 300 steps

This is the Gate E validation. `tok/s` from the SUMMARY line is the number we need: measured throughput → real MFU → corrected GPU-hour estimates for the 125M/2.5B-token and 350M targets.

Single-GPU run (`cuda:0`). The optional Run C cell covers the 2-GPU path separately.

In [ ]:
import subprocess
r = subprocess.run(['python', 'src/train.py', '--steps', '300', '--t-max', '300',
                    '--device', 'cuda', '--d-model', '512', '--n-layers', '8',
                    '--n-heads', '8', '--n-kv-heads', '4',
                    '--batch-size', '16', '--seq-len', '512',
                    '--ckpt-dir', 'checkpoints/gpu_smoke_B',
                    '--log-every', '50', '--ckpt-every', '300', '--val-every', '100'],
                   capture_output=True, text=True)
print(r.stdout[-1200:])
assert 'device=cuda' in r.stdout, 'SILENT CPU FALLBACK? device=cuda not in log'
assert r.returncode == 0, 'Run B failed:\n' + r.stderr[-2000:]
print('RUN B OK')


## Run C (optional) — two-GPU DDP smoke

Only if this session exposes **2 GPUs**. Compares `torchrun --nproc_per_node=2` DDP throughput against Run B's single-GPU number. DDP averages gradients across ranks, so one optimizer step covers a 2× global batch — the cell reports aggregate tok/s, per-GPU tok/s, scaling efficiency vs 2× single-GPU, and the implied communication overhead. If it fails, Runs A/B still stand on their own.

In [ ]:
import subprocess
n = torch.cuda.device_count()
if n < 2:
    print(f'only {n} GPU visible — skipping Run C (single-GPU session)')
else:
    r = subprocess.run(['torchrun', '--nproc_per_node=2', 'src/train.py',
                        '--ddp', '--device', 'cuda',
                        '--steps', '40', '--t-max', '40',
                        '--d-model', '512', '--n-layers', '8',
                        '--n-heads', '8', '--n-kv-heads', '4',
                        '--batch-size', '16', '--seq-len', '512',
                        '--ckpt-dir', 'checkpoints/gpu_smoke_C',
                        '--log-every', '10', '--ckpt-every', '40'],
                       capture_output=True, text=True)
    print(r.stdout[-1500:])
    assert r.returncode == 0, 'Run C failed:\n' + r.stderr[-2000:]
    print('RUN C OK — DDP on 2 GPUs executed')


## Results → Gate E check

**Methodology (explicit):**
- *Measured:* tok/s, elapsed time, peak GPU memory, losses — from each run's `last_summary.json`.
- *Estimated:* FLOPs/token ≈ 6N + 12·L·d·S (Kaplan 6N plus the attention term; documented, ±10% methodology band). N is computed exactly from the model config, not rounded.
- *Hardware peak:* we train in **FP32** (no AMP), so the comparable peak is the T4's **FP32 8.1 TFLOPS** — not the 65 TFLOPS FP16-tensor figure. (P100 FP32: 9.3 TFLOPS.)
- MFU = measured tok/s × FLOPs/token ÷ FP32 peak. Uncertainty: methodology band plus short-run measurement noise.
- The 125M projection scales the *measured* MFU; it inherits the same uncertainty — treat it as an order-of-magnitude check on EXP-003's 23–32 T4-hours, not a promise.

In [ ]:
import glob, sys
sys.path.insert(0, 'src')
from model import ForgeConfig

# Hardware (measured)
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
vram = [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())]
FP32_PEAK = {'T4': 8.1e12, 'P100': 9.3e12}  # FP32 peak FLOPS — we train FP32 (no AMP)
peak = None
for name in gpu_names:
    for k, v in FP32_PEAK.items():
        if k in name:
            peak = v

def flops_per_token(d_model, n_layers):
    cfg = ForgeConfig(vocab_size=256, d_model=d_model, n_layers=n_layers,
                      n_heads=8, n_kv_heads=4, max_seq_len=513)
    N = cfg.param_count()['total']
    attn = 12 * n_layers * d_model * 512  # attention term at seq_len 512
    return 6 * N + attn, N

results = {}
run_cfg = {
    'gpu_smoke_A': dict(tag='A-tiny-1M', d_model=128, n_layers=4, steps=40, batch=8, seq=64, gpus=1),
    'gpu_smoke_B': dict(tag='B-25M', d_model=512, n_layers=8, steps=300, batch=16, seq=512, gpus=1),
    'gpu_smoke_C': dict(tag='C-25M-ddp2', d_model=512, n_layers=8, steps=40, batch=16, seq=512, gpus=2),
}
for tag, rc in run_cfg.items():
    p = f'checkpoints/{tag}/last_summary.json'
    try:
        s = json.load(open(p))
    except FileNotFoundError:
        results[rc['tag']] = {'status': 'not run'}
        continue
    fpt, N = flops_per_token(rc['d_model'], rc['n_layers'])
    tps = s['tok_per_s']
    mfu = tps * fpt / peak if peak else None
    results[rc['tag']] = {
        'status': 'ok', 'config': rc, 'params_exact': N,
        'elapsed_s': s['elapsed_s'], 'tok_per_s': tps,
        'peak_gpu_mem_mb': s.get('peak_gpu_mem_mb'),
        'final_train_loss': s['final_train_loss'],
        'final_val_loss': s['final_val_loss'],
        'flops_per_token_est': fpt, 'mfu_fp32': round(mfu, 3) if mfu else None,
    }
    print(rc['tag'], '-> tok/s', tps, '| MFU(fp32) ~', round(mfu, 3) if mfu else None,
          '| peak_mem_mb', s.get('peak_gpu_mem_mb'))

# 125M projection at measured MFU (order-of-magnitude check on EXP-003)
b = results.get('B-25M', {})
if b.get('mfu_fp32') and peak:
    fpt125, _ = flops_per_token(768, 18)  # FORGE-125M config
    hrs = b['tok_per_s'] and (fpt125 * 2.5e9 / (peak * b['mfu_fp32']) / 3600)
    print(f"125M/2.5B tokens at B's measured MFU: ~{hrs:.1f} GPU-hours (EXP-003 said 23-32)")
    b['proj_125m_2p5B_gpu_hours'] = round(hrs, 1)

payload = {
    'notebook': 'forge_m5_gpu_smoke',
    'pinned_commit': FORGE_COMMIT, 'actual_head': actual,
    'hardware': {'gpu_names': gpu_names, 'gpu_count': len(gpu_names),
                 'vram_total_mb': [round(v/1e6, 1) for v in vram],
                 'torch': torch.__version__, 'cuda_torch': torch.version.cuda,
                 'python': sys.version.split()[0],
                 'fp32_peak_flops_used': peak},
    'methodology': ('measured: tok/s, elapsed, peak mem, losses. '
                    'estimated: FLOPs/token = 6N + 12*L*d*S (±10% band). '
                    'MFU vs FP32 peak (training is FP32, no AMP).'),
    'runs': results,
}
json.dump(payload, open('/kaggle/working/gpu_smoke_results.json', 'w'), indent=2)
print('saved /kaggle/working/gpu_smoke_results.json  <- download this as M5 evidence')


## After the run

1. **Download** `gpu_smoke_results.json` from the notebook output (right panel → `/kaggle/working`).
2. Paste the printed tok/s + MFU numbers back to Khushi — the Gate E table in EXP-003 gets revised with real numbers.
3. If Run B's MFU lands in the 25–35% band, the M5 pretraining plan (125M v0 ≈ 23–32 T4-hours) is validated and the corpus decision can proceed with confidence. If it's far off, we revise before burning quota.

**Troubleshooting:**
- `CUDA out of memory` → halve `--batch-size` and re-run B (note the new batch size in your report).
- `torch.use_deterministic_algorithms` error on GPU → report the exact op; the smoke result then carries a determinism caveat.
- Quota/queue messages from Kaggle are account-side, not a FORGE bug — just re-run later.